# Local AI Sales Intelligence

A complete source walkthrough in ten chapters. Each chapter contains explanations, full file-writing code cells, runnable fictional-data examples and a verification checkpoint.

Start Jupyter from this project using its virtual environment. Read `README.md` first. The delivered source files are already present. Source cells reconstruct a separate copy in `notebook_project/`; demonstrations import the supplied, tested package.

All demonstrations use clearly fictional transactions. Model and n8n calls are opt-in. `Run All` does not install system software, download models, publish workflows, or start services. To review one chapter at a time, use the individual notebooks and pause at each checkpoint.

The original prompt's ten topics are retained. Basic deterministic preparation comes before optional local models, and the complete web API is assembled after its modules exist.


In [ ]:
from pathlib import Path
import os, sys, json

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), "Open this notebook from the supplied project or its notebooks directory."
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
# Source-writing cells reconstruct a separate copy and preserve the supplied project.
BUILD_ROOT = ROOT / "notebook_project"
BUILD_ROOT.mkdir(exist_ok=True)
for folder in ["sales_intelligence", "scripts", "tests", "workflows", "frontend/src", "docs"]:
    (BUILD_ROOT / folder).mkdir(parents=True, exist_ok=True)
RUN_LOCAL_MODELS = False  # opt in only after installing local Ollama models
RUN_N8N = False          # opt in only after configuring/publishing the local workflows
print("Python:", sys.version.split()[0])
print("Source reconstruction:", BUILD_ROOT)


## Chapter 7 · n8n automation

Use n8n to submit a sales question and retrieve task status. A separate inactive schedule can submit a daily dashboard job. The backend owns data processing and agent state; n8n owns triggers and API connections.

### Code cell · `scripts/build_workflows.py`

This is the complete source file. Running the cell writes a separate reconstruction under `notebook_project/`.

In [ ]:
%%writefile "{BUILD_ROOT / 'scripts/build_workflows.py'}"
"""Generate importable n8n workflows. No secrets and no executable Code nodes."""

import json
from pathlib import Path
from uuid import NAMESPACE_URL, uuid5

ROOT = Path(__file__).resolve().parents[1]


def node(name, type_name, version, parameters, position, credential=None):
    result = {
        "id": str(uuid5(NAMESPACE_URL, name)),
        "name": name,
        "type": "n8n-nodes-base." + type_name,
        "typeVersion": version,
        "parameters": parameters,
        "position": position,
    }
    if credential:
        result["credentials"] = {"httpHeaderAuth": {"id": "configure-after-import", "name": credential}}
    if type_name == "webhook":
        result["webhookId"] = str(uuid5(NAMESPACE_URL, parameters["path"]))
    return result


def request(name, method, url, position, body=None):
    parameters = {
        "method": method,
        "url": url,
        "authentication": "genericCredentialType",
        "genericAuthType": "httpHeaderAuth",
        "options": {"timeout": 15000},
    }
    if body is not None:
        parameters.update(sendBody=True, specifyBody="json", jsonBody=body)
    return node(name, "httpRequest", 4.2, parameters, position, "Sales backend API")


def workflow(name, nodes, edges):
    connections = {}
    for left, right in edges:
        connections[left] = {"main": [[{"node": right, "type": "main", "index": 0}]]}
    return {
        "name": name,
        "nodes": nodes,
        "connections": connections,
        "active": False,
        "settings": {
            "executionOrder": "v1",
            "timezone": "Asia/Tehran",
            "saveDataSuccessExecution": "none",
            "saveDataErrorExecution": "none",
            "saveManualExecutions": False,
        },
        "pinData": {},
        "tags": [],
    }


def build():
    directory = ROOT / "workflows"
    directory.mkdir(exist_ok=True)
    nodes = [
        node(
            "Sales question",
            "webhook",
            2,
            {
                "httpMethod": "POST",
                "path": "sales-advisor",
                "authentication": "headerAuth",
                "responseMode": "responseNode",
                "options": {},
            },
            [0, 0],
            "Sales webhook access",
        ),
        request(
            "Submit local agent",
            "POST",
            "={{ 'http://api:8000/api/chat/' + encodeURIComponent($json.body.dataset_id) }}",
            [260, 0],
            "={{ { question: $json.body.question, filters: $json.body.filters ?? {}, use_llm: $json.body.use_llm === true } }}",
        ),
        node(
            "Return task receipt",
            "respondToWebhook",
            1.4,
            {"respondWith": "json", "responseBody": "={{ $json }}", "options": {"responseCode": 202}},
            [520, 0],
        ),
    ]
    chat = workflow(
        "Local sales advisor - submit question",
        nodes,
        [("Sales question", "Submit local agent"), ("Submit local agent", "Return task receipt")],
    )
    nodes = [
        node(
            "Task status",
            "webhook",
            2,
            {
                "httpMethod": "GET",
                "path": "sales-status",
                "authentication": "headerAuth",
                "responseMode": "responseNode",
                "options": {},
            },
            [0, 0],
            "Sales webhook access",
        ),
        request(
            "Read local task",
            "GET",
            "={{ 'http://api:8000/api/tasks/' + encodeURIComponent($json.query.task_id) }}",
            [260, 0],
        ),
        node(
            "Return status and evidence",
            "respondToWebhook",
            1.4,
            {"respondWith": "json", "responseBody": "={{ $json }}", "options": {"responseCode": 200}},
            [520, 0],
        ),
    ]
    status = workflow(
        "Local sales advisor - task status",
        nodes,
        [("Task status", "Read local task"), ("Read local task", "Return status and evidence")],
    )
    nodes = [
        node(
            "Daily local report",
            "scheduleTrigger",
            1.2,
            {"rule": {"interval": [{"field": "cronExpression", "expression": "0 8 * * *"}]}},
            [0, 0],
        ),
        node(
            "Choose approved dataset",
            "set",
            3.4,
            {
                "assignments": {
                    "assignments": [
                        {
                            "id": "dataset",
                            "name": "dataset_id",
                            "type": "string",
                            "value": "REPLACE_WITH_APPROVED_DATASET_UUID",
                        }
                    ]
                },
                "options": {},
            },
            [250, 0],
        ),
        request(
            "Queue dashboard report",
            "POST",
            "={{ 'http://api:8000/api/analyze/' + encodeURIComponent($json.dataset_id) }}",
            [500, 0],
            "={{ {} }}",
        ),
    ]
    schedule = workflow(
        "Local sales advisor - daily dashboard",
        nodes,
        [
            ("Daily local report", "Choose approved dataset"),
            ("Choose approved dataset", "Queue dashboard report"),
        ],
    )
    for filename, value in [
        ("sales_advisor.json", chat),
        ("task_status.json", status),
        ("daily_report.json", schedule),
    ]:
        (directory / filename).write_text(json.dumps(value, indent=2, ensure_ascii=False) + "\n")
    print(
        "Generated 3 inactive workflows; assign header credentials and verify local API URLs before publishing."
    )


if __name__ == "__main__":
    build()


### Code cell · `workflows/sales_advisor.json`

This is the complete source file. Running the cell writes a separate reconstruction under `notebook_project/`.

In [ ]:
%%writefile "{BUILD_ROOT / 'workflows/sales_advisor.json'}"
{
  "name": "Local sales advisor - submit question",
  "nodes": [
    {
      "id": "c03ea51c-e1bb-59bf-a0fe-e200db4963dd",
      "name": "Sales question",
      "type": "n8n-nodes-base.webhook",
      "typeVersion": 2,
      "parameters": {
        "httpMethod": "POST",
        "path": "sales-advisor",
        "authentication": "headerAuth",
        "responseMode": "responseNode",
        "options": {}
      },
      "position": [
        0,
        0
      ],
      "credentials": {
        "httpHeaderAuth": {
          "id": "configure-after-import",
          "name": "Sales webhook access"
        }
      },
      "webhookId": "d689c234-30fa-5d4e-b48a-c532c6d1e4ee"
    },
    {
      "id": "8102b326-185a-52c9-bdfb-be737c22e8ae",
      "name": "Submit local agent",
      "type": "n8n-nodes-base.httpRequest",
      "typeVersion": 4.2,
      "parameters": {
        "method": "POST",
        "url": "={{ 'http://api:8000/api/chat/' + encodeURIComponent($json.body.dataset_id) }}",
        "authentication": "genericCredentialType",
        "genericAuthType": "httpHeaderAuth",
        "options": {
          "timeout": 15000
        },
        "sendBody": true,
        "specifyBody": "json",
        "jsonBody": "={{ { question: $json.body.question, filters: $json.body.filters ?? {}, use_llm: $json.body.use_llm === true } }}"
      },
      "position": [
        260,
        0
      ],
      "credentials": {
        "httpHeaderAuth": {
          "id": "configure-after-import",
          "name": "Sales backend API"
        }
      }
    },
    {
      "id": "f266d756-df2a-5ad6-8414-cc2dc3770927",
      "name": "Return task receipt",
      "type": "n8n-nodes-base.respondToWebhook",
      "typeVersion": 1.4,
      "parameters": {
        "respondWith": "json",
        "responseBody": "={{ $json }}",
        "options": {
          "responseCode": 202
        }
      },
      "position": [
        520,
        0
      ]
    }
  ],
  "connections": {
    "Sales question": {
      "main": [
        [
          {
            "node": "Submit local agent",
            "type": "main",
            "index": 0
          }
        ]
      ]
    },
    "Submit local agent": {
      "main": [
        [
          {
            "node": "Return task receipt",
            "type": "main",
            "index": 0
          }
        ]
      ]
    }
  },
  "active": false,
  "settings": {
    "executionOrder": "v1",
    "timezone": "Asia/Tehran",
    "saveDataSuccessExecution": "none",
    "saveDataErrorExecution": "none",
    "saveManualExecutions": false
  },
  "pinData": {},
  "tags": []
}


### Code cell · `workflows/task_status.json`

This is the complete source file. Running the cell writes a separate reconstruction under `notebook_project/`.

In [ ]:
%%writefile "{BUILD_ROOT / 'workflows/task_status.json'}"
{
  "name": "Local sales advisor - task status",
  "nodes": [
    {
      "id": "ebbd325c-c3f5-52db-9b74-423bc5b0b376",
      "name": "Task status",
      "type": "n8n-nodes-base.webhook",
      "typeVersion": 2,
      "parameters": {
        "httpMethod": "GET",
        "path": "sales-status",
        "authentication": "headerAuth",
        "responseMode": "responseNode",
        "options": {}
      },
      "position": [
        0,
        0
      ],
      "credentials": {
        "httpHeaderAuth": {
          "id": "configure-after-import",
          "name": "Sales webhook access"
        }
      },
      "webhookId": "99815ba0-2331-5f9b-bbd4-93f58dabd1e9"
    },
    {
      "id": "9c12a5c5-465a-5641-b65e-ede73af3beda",
      "name": "Read local task",
      "type": "n8n-nodes-base.httpRequest",
      "typeVersion": 4.2,
      "parameters": {
        "method": "GET",
        "url": "={{ 'http://api:8000/api/tasks/' + encodeURIComponent($json.query.task_id) }}",
        "authentication": "genericCredentialType",
        "genericAuthType": "httpHeaderAuth",
        "options": {
          "timeout": 15000
        }
      },
      "position": [
        260,
        0
      ],
      "credentials": {
        "httpHeaderAuth": {
          "id": "configure-after-import",
          "name": "Sales backend API"
        }
      }
    },
    {
      "id": "d0d34909-9f63-5c10-8bca-b31cc749f541",
      "name": "Return status and evidence",
      "type": "n8n-nodes-base.respondToWebhook",
      "typeVersion": 1.4,
      "parameters": {
        "respondWith": "json",
        "responseBody": "={{ $json }}",
        "options": {
          "responseCode": 200
        }
      },
      "position": [
        520,
        0
      ]
    }
  ],
  "connections": {
    "Task status": {
      "main": [
        [
          {
            "node": "Read local task",
            "type": "main",
            "index": 0
          }
        ]
      ]
    },
    "Read local task": {
      "main": [
        [
          {
            "node": "Return status and evidence",
            "type": "main",
            "index": 0
          }
        ]
      ]
    }
  },
  "active": false,
  "settings": {
    "executionOrder": "v1",
    "timezone": "Asia/Tehran",
    "saveDataSuccessExecution": "none",
    "saveDataErrorExecution": "none",
    "saveManualExecutions": false
  },
  "pinData": {},
  "tags": []
}


### Code cell · `workflows/daily_report.json`

This is the complete source file. Running the cell writes a separate reconstruction under `notebook_project/`.

In [ ]:
%%writefile "{BUILD_ROOT / 'workflows/daily_report.json'}"
{
  "name": "Local sales advisor - daily dashboard",
  "nodes": [
    {
      "id": "e42ed7cc-c9ce-534e-ba17-cb153c2dba88",
      "name": "Daily local report",
      "type": "n8n-nodes-base.scheduleTrigger",
      "typeVersion": 1.2,
      "parameters": {
        "rule": {
          "interval": [
            {
              "field": "cronExpression",
              "expression": "0 8 * * *"
            }
          ]
        }
      },
      "position": [
        0,
        0
      ]
    },
    {
      "id": "508d1fef-b815-50b1-a774-5153471417ef",
      "name": "Choose approved dataset",
      "type": "n8n-nodes-base.set",
      "typeVersion": 3.4,
      "parameters": {
        "assignments": {
          "assignments": [
            {
              "id": "dataset",
              "name": "dataset_id",
              "type": "string",
              "value": "REPLACE_WITH_APPROVED_DATASET_UUID"
            }
          ]
        },
        "options": {}
      },
      "position": [
        250,
        0
      ]
    },
    {
      "id": "ffbe719a-3827-5ba5-b232-da1018a28a65",
      "name": "Queue dashboard report",
      "type": "n8n-nodes-base.httpRequest",
      "typeVersion": 4.2,
      "parameters": {
        "method": "POST",
        "url": "={{ 'http://api:8000/api/analyze/' + encodeURIComponent($json.dataset_id) }}",
        "authentication": "genericCredentialType",
        "genericAuthType": "httpHeaderAuth",
        "options": {
          "timeout": 15000
        },
        "sendBody": true,
        "specifyBody": "json",
        "jsonBody": "={{ {} }}"
      },
      "position": [
        500,
        0
      ],
      "credentials": {
        "httpHeaderAuth": {
          "id": "configure-after-import",
          "name": "Sales backend API"
        }
      }
    }
  ],
  "connections": {
    "Daily local report": {
      "main": [
        [
          {
            "node": "Choose approved dataset",
            "type": "main",
            "index": 0
          }
        ]
      ]
    },
    "Choose approved dataset": {
      "main": [
        [
          {
            "node": "Queue dashboard report",
            "type": "main",
            "index": 0
          }
        ]
      ]
    }
  },
  "active": false,
  "settings": {
    "executionOrder": "v1",
    "timezone": "Asia/Tehran",
    "saveDataSuccessExecution": "none",
    "saveDataErrorExecution": "none",
    "saveManualExecutions": false
  },
  "pinData": {},
  "tags": []
}


### Run the chapter example

In [ ]:
workflows = {}
for path in (ROOT / "workflows").glob("*.json"):
    flow = json.loads(path.read_text())
    assert flow["active"] is False
    workflows[path.name] = {"name": flow["name"], "nodes": [node["name"] for node in flow["nodes"]]}
display(workflows)
if RUN_N8N:
    import getpass, httpx
    base = "http://localhost:5678"
    webhook_key = getpass.getpass("Local n8n webhook key: ")
    approved_dataset_id = input("Prepared API dataset UUID, not the notebook demonstration UUID: ").strip()
    response = httpx.post(base + "/webhook/sales-advisor", headers={"X-Webhook-Key": webhook_key},
        json={"dataset_id": approved_dataset_id, "question": "Which products sell best?", "use_llm": False},
        timeout=20, trust_env=False)
    response.raise_for_status()
    display(response.json())
else:
    print("Live n8n call not run. Import the JSON files and configure header credentials using docs/n8n_setup.md.")


### Review checkpoint

Use separate credentials for n8n webhook access and the backend API. Never export secrets into workflow JSON. Docker URLs use http://api:8000; a fully native setup uses http://127.0.0.1:8000. No workflow sends messages or publishes itself.

Review this result before moving to the next chapter.